In [6]:
import numpy as np

TARGETS = ["Theta"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [10]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 3

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [11]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
display(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====


,Target,Dataset,Best_Model,Neurons,Best_R2
0,Theta,Train_1,model_32_16_8_Ld0.7_Lp0.3_r0.01_seed0,"[32, 16, 8]",[0.9279762550595982]
1,Theta,Train_2,model_264_128_Ld0.7_Lp0.3_r0.5_seed0,"[264, 128]",[0.868291627641418]
2,Theta,Val,model_264_Ld0.3_Lp0.7_r0.9_seed1,"[16, 8, 4]",[0.752384537930381]
3,Theta,Test_1,model_32_16_8_Ld0.3_Lp0.7_r0.01_seed0,"[32, 16, 8]",[0.9203841418818819]
4,Theta,Test_2,model_16_8_4_Ld0.3_Lp0.7_r0.5_seed2,"[16, 8, 4]",[0.8971528709909345]
5,Theta,Test_3,model_16_8_Ld0.3_Lp0.7_r0.01_seed0,"[16, 8]",[0.7877008399175166]


In [12]:
df_summary_top10["bestR2"] = best_only_df["Best_R2"].str.strip("[]").astype(float)
df_summary_top10["Neurons"] = best_only_df["Neurons"]
display(df_summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,Theta,0.918750,0.008696,0.018547,0.001985,0.927976,"[32, 16, 8]"
1,Train_2,Theta,0.866512,0.001961,0.030476,0.000448,0.868292,"[264, 128]"
2,Val,Theta,0.691526,0.052729,0.100957,0.017257,0.752385,"[16, 8, 4]"
3,Test_1,Theta,0.911522,0.011215,0.020325,0.002576,0.920384,"[32, 16, 8]"
4,Test_2,Theta,0.893203,0.003549,0.025328,0.000842,0.897153,"[16, 8, 4]"
5,Test_3,Theta,0.773632,0.014805,0.074418,0.004867,0.787701,"[16, 8]"


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted